# XLM-RoBERTa Sentiment Model (PyTorch + Transformers)


## 1. Get the data

In [ ]:
!git clone https://github.com/MennaMehelib/Multilingual-Customer-Sentiment-Analysis-Platform.git

In [ ]:
%cd Multilingual-Customer-Sentiment-Analysis-Platform

In [ ]:
import pandas as pd

train_df = pd.read_csv("data/processed/splits/train.csv")
val_df   = pd.read_csv("data/processed/splits/validation.csv")
test_df  = pd.read_csv("data/processed/splits/test.csv")

In [ ]:
labels   = ["negative", "neutral", "positive"]
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

In [ ]:
print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

## 2. Settings + Tokenizer + DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LEN    = 64
BATCH_SIZE = 64
EPOCHS     = 3
LR         = 2e-5

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class ReviewsDataset(Dataset):
    def __init__(self, df):
        self.encodings = tokenizer(
            df["cleaned_text"].fillna("").astype(str).tolist(),
            truncation=True,
            max_length=MAX_LEN
        )
        self.labels = df["label"].map(label2id).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: torch.tensor(value[i]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

In [ ]:
def collate(batch):
    labels = torch.stack([item.pop("labels") for item in batch])
    enc = tokenizer.pad(batch, padding=True, return_tensors="pt")
    enc["labels"] = labels
    return enc

In [ ]:
train_loader = DataLoader(ReviewsDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)
val_loader   = DataLoader(ReviewsDataset(val_df),   batch_size=128, collate_fn=collate, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ReviewsDataset(test_df),  batch_size=128, collate_fn=collate, num_workers=2, pin_memory=True)

In [ ]:
# Example of tokenization
print(tokenizer.tokenize("The service was really good "))

## 3. Model + Training

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

In [ ]:
torch.manual_seed(42)

# 1. Load the model and move it to GPU directly
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id).to(device)

In [ ]:
optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler      = torch.amp.GradScaler()   # mixed precision = faster on GPU

In [ ]:
def evaluate(loader):
    """Returns true labels, predicted labels and probabilities."""
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**batch).logits
            all_probs.append(logits.float().softmax(-1).cpu())
            all_labels.append(batch["labels"].cpu())
    probs = torch.cat(all_probs).numpy()
    return torch.cat(all_labels).numpy(), probs.argmax(1), probs

In [ ]:
def get_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return  {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f1}

In [ ]:
best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    y_true, y_pred, _ = evaluate(val_loader)
    m = get_metrics(y_true, y_pred)
    print(f"Epoch {epoch + 1}: train_loss={total_loss / len(train_loader):.4f} | "
          f"val_accuracy={m['accuracy']:.4f} | val_f1={m['f1']:.4f}")

    # keep the best model only
    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        model.save_pretrained("saved_model")
        tokenizer.save_pretrained("saved_model")
        print("  -> best model saved")

## 4. Evaluate on the test set

In [ ]:
import os, json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
os.makedirs("results", exist_ok=True)

In [ ]:
# load the best saved model
model = AutoModelForSequenceClassification.from_pretrained("saved_model").to(device)
y_true, y_pred, probs = evaluate(test_loader)

In [ ]:
test_metrics = get_metrics(y_true, y_pred)
print(test_metrics)
print(classification_report(y_true, y_pred, target_names=labels, digits=4))
json.dump(test_metrics, open("results/metrics.json", "w"), indent=2)

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=labels).plot(cmap="Blues", values_format="d")
plt.title("XLM-RoBERTa - Test Confusion Matrix")
plt.savefig("results/confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# all test predictions (for Person 4 error analysis)
test_df["predicted"]  = [id2label[i] for i in y_pred]
test_df["confidence"] = probs.max(1).round(4)
test_df[["cleaned_text", "language", "label", "predicted", "confidence"]].to_csv(
    "results/test_predictions.csv", index=False, encoding="utf-8-sig")

## 5. Sample predictions

In [ ]:
def predict(texts):
    model.eval()
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = model(**enc).logits.softmax(-1).cpu()
    return pd.DataFrame({
        "text": texts,
        "prediction": [id2label[int(i)] for i in probs.argmax(1)],
        "confidence": probs.max(1).values.numpy().round(3),
    })

In [ ]:
samples = predict([
    "The service was excellent and the staff were very friendly!",
    "Terrible experience, the product arrived broken.",
    "It's okay, nothing special.",
    "الخدمة ممتازة والتوصيل كان سريع جدا",
    "المنتج وحش جدا ومحدش رد عليا",
    "这个产品质量很好，我很满意",
    "还行吧，一般般",
])
samples.to_csv("results/sample_predictions.csv", index=False, encoding="utf-8-sig")
samples

## 6. Files to run the model

In [ ]:
%%writefile saved_model/predict.py
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = os.path.dirname(os.path.abspath(__file__))
MAX_LEN = 64

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()


def predict(texts):
    enc = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = model(**enc).logits.softmax(-1).cpu()
    return [(t, model.config.id2label[int(i)], float(p)) for t, i, p in zip(texts, probs.argmax(1), probs.max(1).values)]


if __name__ == "__main__":
    for text, label, conf in predict(sys.argv[1:]):
        print(f"{label} ({conf:.2f}): {text}")

In [ ]:
%%writefile saved_model/requirements.txt
torch
transformers

## 7. Download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("saved_model", "zip", "saved_model")
shutil.make_archive("results", "zip", "results")
files.download("saved_model.zip")
files.download("results.zip")